# INTRODUÇÃO

In [61]:
# Importando as bibliotecas

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
combine = [train, test]

# DESCRIÇÃO DOS DADOS

In [62]:
train.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [63]:
train.info()
print('-'*40)
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
----------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Passenger

# LIMPEZA E TRATAMENTO

In [64]:
# Substituindo os valores nulos na coluna Age pela mediana de todas as idades

Age_mediana = train['Age'].median()
train['Age'] = train['Age'].fillna(Age_mediana)
test['Age'] = test['Age'].fillna(Age_mediana)

# Substituindo os valores nulos nas colunas Cabin, Embarked e Fare

for dataset in combine:
    dataset['Cabin'] = dataset['Cabin'].fillna('Desconhecido')
    dataset['Embarked'] = dataset['Embarked'].fillna(dataset['Embarked'].mode()[0])
    dataset['Fare'] = dataset['Fare'].fillna(dataset.groupby('Pclass')['Fare'].transform('median'))

train.info()
print('-'*40)
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          891 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        891 non-null    object 
 11  Embarked     891 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
----------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Passenger

# FEATURING ENGINEERING

In [65]:
# Criando a coluna Título contendo os títulos presentes em cada nome na coluna Name

for dataset in combine:
    dataset['Título'] = dataset.Name.str.extract(' ([A-Za-z]+)\.', expand=False)

pd.crosstab(train['Título'], train['Sex'])

Sex,female,male
Título,,
Capt,0,1
Col,0,2
Countess,1,0
Don,0,1
Dr,1,6
Jonkheer,0,1
Lady,1,0
Major,0,2
Master,0,40


In [66]:
# Filtrando os principais títulos presentes

for dataset in combine:
    dataset['Título'] = dataset['Título'].replace(['Capt', 'Col', 'Countess', 'Don', 'Dr', 'Jonkheer', 'Lady',\
                                                  'Major', 'Mlle', 'Mme', 'Ms', 'Rev', 'Sir', 'Dona'], 'Rare')
train[['Título', 'Survived']].groupby(['Título'], as_index = False).mean()

,Título,Survived
0,Master,0.575000
1,Miss,0.697802
2,Mr,0.156673
3,Mrs,0.792000
4,Rare,0.444444


In [67]:
# Ajustando a coluna Título para a aplicação dos algoritmos de Machine Learning

for dataset in combine:
    if dataset['Título'].dtype == 'object':
        dataset['Título'] = dataset['Título'].map({
        'Master': 1,
        'Miss': 2,
        'Mr': 3,
        'Mrs': 4,
        'Rare': 5})

train.tail(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Título
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,Desconhecido,S,5
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S,2
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,28.0,1,2,W./C. 6607,23.45,Desconhecido,S,2
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C,3
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,Desconhecido,Q,3


In [68]:
# Criando a coluna família para identificar os grupos familiares e quem viajava sozinho

for dataset in combine:
    dataset['Família'] = dataset['SibSp'] + dataset['Parch'] + 1

train[['Família', 'Survived']].groupby(['Família'], as_index=False).mean()

,Família,Survived
0,1,0.303538
1,2,0.552795
2,3,0.578431
3,4,0.724138
4,5,0.200000
5,6,0.136364
6,7,0.333333
7,8,0.000000
8,11,0.000000


In [69]:
# Ajustando a coluna Embarked para a aplicação dos algoritmos de Machine Learning

for dataset in combine:
    if dataset['Embarked'].dtype == 'object':
        dataset['Embarked'] = dataset['Embarked'].map({
        'S': 0,
        'C': 1,
        'Q': 2})

In [70]:
# Ajustando a coluna Sex para a aplicação dos algoritmos de Machine Learning

for dataset in combine:
    if dataset['Sex'].dtype == 'object':
        dataset['Sex'] = dataset['Sex'].map({
        'female': 1,
        'male': 0})

In [71]:
# Criando a melhor forma de agrupar as pessoas a partir das respectivas idades

train['Age_Class'] = pd.cut(train['Age'], 5)
train[['Age_Class', 'Survived']].groupby(['Age_Class'], as_index = False, observed = True).mean()

,Age_Class,Survived
0,"(0.34, 16.336]",0.550000
1,"(16.336, 32.252]",0.344168
2,"(32.252, 48.168]",0.404255
3,"(48.168, 64.084]",0.434783
4,"(64.084, 80.0]",0.090909


In [72]:
# Ajustando a coluna Age para a aplicação dos algoritmos de Machine Learning

for dataset in combine:
    if dataset['Age'].dtype == 'float':
        dataset.loc[dataset['Age'] <= 16, 'Age'] = 0
        dataset.loc[(dataset['Age'] > 16) & (dataset['Age'] <=32), 'Age'] = 1
        dataset.loc[(dataset['Age'] > 32) & (dataset['Age'] <=48), 'Age'] = 2
        dataset.loc[(dataset['Age'] > 48) & (dataset['Age'] <=64), 'Age'] = 3
        dataset.loc[dataset['Age'] > 64, 'Age'] = 4
        dataset['Age'] = dataset['Age'].astype('int')

In [73]:
# Criando a melhor forma de agrupar as pessoas a partir das respectivas tarifas pagas

train['Fare_Class'] = pd.qcut(train['Fare'], 4)
train[['Fare_Class', 'Survived']].groupby(['Fare_Class'], as_index = False, observed = True).mean()

,Fare_Class,Survived
0,"(-0.001, 7.91]",0.197309
1,"(7.91, 14.454]",0.303571
2,"(14.454, 31.0]",0.454955
3,"(31.0, 512.329]",0.581081


In [74]:
# Ajustando a coluna Fare para a aplicação dos algoritmos de Machine Learning

for dataset in combine:
    if dataset['Fare'].dtype == 'float':
        dataset.loc[dataset['Fare'] <= 7.91, 'Fare'] = 0
        dataset.loc[(dataset['Fare'] > 7.91) & (dataset['Fare'] <= 14.454), 'Fare'] = 1
        dataset.loc[(dataset['Fare'] > 14.454) & (dataset['Fare'] <= 31), 'Fare'] = 2
        dataset.loc[dataset['Fare'] > 31, 'Fare'] = 3
        dataset['Fare'] = dataset['Fare'].astype('int')

# MODELAGEM

In [75]:
# Separando em datasets de treino e teste

train_df = train[['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Título', 'Família']]
test_df = test[['PassengerId', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'Título', 'Família']]

In [76]:
# Separando X e y

X = train_df.drop("Survived", axis=1)
y = train_df["Survived"]
X_test  = test_df.drop("PassengerId", axis=1).copy()
X.shape, y.shape, X_test.shape

((891, 7), (891,), (418, 7))

In [77]:
# Split

X_train, X_valid, Y_train, Y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [78]:
# Regressão Logística

logreg = LogisticRegression(random_state=42)
logreg.fit(X_train, Y_train)
Y_pred = logreg.predict(X_test)
acc_log = round(logreg.score(X_valid, Y_valid) * 100, 2)
acc_log

79.89

In [79]:
# SVC

svc = SVC()
svc.fit(X_train, Y_train)
Y_pred = svc.predict(X_test)
acc_svc = round(svc.score(X_valid, Y_valid) * 100, 2)
acc_svc

82.68

In [80]:
# K Neighbors

knn = KNeighborsClassifier(n_neighbors = 3)
knn.fit(X_train, Y_train)
Y_pred = knn.predict(X_test)
acc_knn = round(knn.score(X_valid, Y_valid) * 100, 2)
acc_knn

80.45

In [81]:
# Gaussian NB

gaussian = GaussianNB()
gaussian.fit(X_train, Y_train)
Y_pred = gaussian.predict(X_test)
acc_gaussian = round(gaussian.score(X_valid, Y_valid) * 100, 2)
acc_gaussian

77.09

In [82]:
# Perceptron

perceptron = Perceptron(random_state=42)
perceptron.fit(X_train, Y_train)
Y_pred = perceptron.predict(X_test)
acc_perceptron = round(perceptron.score(X_valid, Y_valid) * 100, 2)
acc_perceptron

74.86

In [83]:
# Linear SVC

linear_svc = LinearSVC()
linear_svc.fit(X_train, Y_train)
Y_pred = linear_svc.predict(X_test)
acc_linear_svc = round(linear_svc.score(X_valid, Y_valid) * 100, 2)
acc_linear_svc

78.21

In [84]:
# SGD

sgd = SGDClassifier(random_state=42)
sgd.fit(X_train, Y_train)
Y_pred = sgd.predict(X_test)
acc_sgd = round(sgd.score(X_valid, Y_valid) * 100, 2)
acc_sgd

80.45

In [85]:
# Árvore de decisão

decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(X_train, Y_train)
Y_pred = decision_tree.predict(X_test)
acc_decision_tree = round(decision_tree.score(X_valid, Y_valid) * 100, 2)
acc_decision_tree

83.24

In [86]:
# Random Forest

random_forest = RandomForestClassifier(n_estimators=100, random_state=42)
random_forest.fit(X_train, Y_train)
Y_pred = random_forest.predict(X_test)
random_forest.score(X_train, Y_train)
acc_random_forest = round(random_forest.score(X_valid, Y_valid) * 100, 2)
acc_random_forest

82.68

# AVALIAÇÃO

In [87]:
# Acurácia

model.fit(X_train, Y_train)

pred = model.predict(X_valid)

accuracy_score(Y_valid, pred)

0.8268156424581006

# CONCLUSÃO

In [88]:
# Verificando o Score de cada modelo

models = pd.DataFrame({
    'Model': ['Support Vector Machines', 'KNN', 'Logistic Regression', 
              'Random Forest', 'Naive Bayes', 'Perceptron', 
              'Stochastic Gradient Decent', 'Linear SVC', 
              'Decision Tree'],
    'Score': [acc_svc, acc_knn, acc_log, 
              acc_random_forest, acc_gaussian, acc_perceptron, 
              acc_sgd, acc_linear_svc, acc_decision_tree]})
models.sort_values(by='Score', ascending=False)

,Model,Score
8,Decision Tree,83.24
0,Support Vector Machines,82.68
3,Random Forest,82.68
1,KNN,80.45
6,Stochastic Gradient Decent,80.45
2,Logistic Regression,79.89
7,Linear SVC,78.21
4,Naive Bayes,77.09
5,Perceptron,74.86
